In [1]:
import os 
import sys

sys.path.append("../../../..")

from src.pricer import *
MarketData.initialize()

QuantCourseBP 6670d3218865c6c04058aa0faec092afa9adc3c2*


In [2]:
und = Stock.TEST_COMPANY
ls = LongShort.LONG
strike = 1.2
expiry = 1
strike_level = strike * MarketData.get_spot()[und]
model = FlatVolModel(und)


contract_opt = EuropeanContract(und, PutCallFwd.CALL, ls, strike_level, expiry)
pricer_opt_an = EuropeanAnalyticPricer(contract_opt, model, Params())
fv_opt_an = pricer_opt_an.calc_fair_value()


mc_params = MCParams(num_of_path=10000,
                     tenor_frequency=0,
                     antithetic=False,
                     standardize=False,
                     control_variate=False,
                     seed=1,
                     evolve_spot_method=MCNumMethod.EXACT)

pricer_opt_mc = GenericMCPricer(contract_opt, model, mc_params)
fv_opt_mc = pricer_opt_mc.calc_fair_value_with_ci()

pricer_opt_mc.params.evolve_spot_method = MCNumMethod.MILSTEIN
pricer_opt_mc.params.tenor_frequency = 4
fv_opt_mc_m = pricer_opt_mc.calc_fair_value_with_ci()


print("OPTION AN: " + str(fv_opt_an))
print("OPTION MC: " + str(fv_opt_mc))
print("OPTION MC Milstein: " + str(fv_opt_mc_m))

OPTION AN: 6.446775044125889
OPTION MC: (6.650668002061888, (6.340481636973081, 6.960854367150695))
OPTION MC Milstein: (6.209709013975387, (5.916888856238308, 6.5025291717124665))


In [11]:
tenor_frequencies = [5, 10, 20, 30, 40, 50]
dt_values = [expiry / n for n in tenor_frequencies]

euler_errors = []
milstein_errors = []

print(f"{'Lépésszám':^15} | {'dt (időlépés)':^15} | {'Euler hiba':^15} | {'Milstein hiba':^15}")
print("-" * 72)

for n, dt in zip(tenor_frequencies, dt_values):
    
    params_exact = MCParams(num_of_path=10000, tenor_frequency=n, seed=1, evolve_spot_method=MCNumMethod.EXACT)
    params_euler = MCParams(num_of_path=10000, tenor_frequency=n, seed=1, evolve_spot_method=MCNumMethod.EULER)
    params_milstein = MCParams(num_of_path=10000, tenor_frequency=n, seed=1, evolve_spot_method=MCNumMethod.MILSTEIN)
    
    method_exact = MCMethodFlatVol(contract_opt, model, params_exact)
    method_euler = MCMethodFlatVol(contract_opt, model, params_euler)
    method_milstein = MCMethodFlatVol(contract_opt, model, params_milstein)
    
    paths_exact = method_exact.simulate_spot_paths()
    paths_euler = method_euler.simulate_spot_paths()
    paths_milstein = method_milstein.simulate_spot_paths()
    
    S_T_exact = paths_exact[:, -1]
    S_T_euler = paths_euler[:, -1]
    S_T_milstein = paths_milstein[:, -1]
    
    err_euler = np.mean(np.abs(S_T_euler - S_T_exact))
    err_milstein = np.mean(np.abs(S_T_milstein - S_T_exact))
    
    euler_errors.append(err_euler)
    milstein_errors.append(err_milstein)
    
    print(f"{n:^15} | {dt:^15.5f} | {err_euler:^15.5f} | {err_milstein:^15.5f}")

   Lépésszám    |  dt (időlépés)  |   Euler hiba    |  Milstein hiba 
------------------------------------------------------------------------
       5        |     0.20000     |     2.13295     |     0.24649    
      10        |     0.10000     |     1.53166     |     0.13166    
      20        |     0.05000     |     1.11806     |     0.06966    
      30        |     0.03333     |     0.91279     |     0.04717    
      40        |     0.02500     |     0.78357     |     0.03593    
      50        |     0.02000     |     0.70010     |     0.02841    
